# 05 — Simple Regression

This notebook continues from `04_eda_visuals.ipynb` and builds a simple OLS regression model predicting `salary_cap_pct` from player performance and usage metrics (e.g. `pie`, `usg_pct`, `ts_pct`, `net_rating`, plus `age` and playing time). Features are standardized before fitting.

The model's residuals (actual minus predicted salary) are used to identify over- and undervalued players: large positive residuals indicate players outperforming their pay, while large negative residuals indicate players overpaid relative to their on-court production. This notebook produces the regression fit, coefficient, and residual figures referenced from `04_eda_visuals.ipynb`, and writes out the model results and top over/under-valued player tables for later use.

## Imports
Load required libraries for data manipulation, numerical computation, and visualization.

In [13]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_project_utils import (
    FIGURES_DIR,
    MODEL_RESULTS_FILE,
    REGRESSION_SUMMARY_FILE,
    REGRESSION_COEFFICIENTS_FILE,
    TOP_OVER_EXPECTED_FILE,
    TOP_UNDER_EXPECTED_FILE,
    JOINED_FILE,
    ensure_figures_dir,
)

## Settings
Set plot style, define the target variable, and list the features used in the regression model.

In [14]:
plt.style.use("ggplot")

TARGET = "salary_cap_pct"

MODEL_FEATURES = [
    'age',
    'gp',
    'min',
    'w_pct',
    'off_rating',
    'def_rating',
    'net_rating',
    'ast_pct',
    'reb_pct',
    'ts_pct',
    'usg_pct',
    'pie',
    'fgm_pg',
    'fg_pct',
]

## Model Functions
Define functions for standardizing features, fitting OLS regression, and computing model metrics (R-squared, RMSE, MAE).

In [15]:
def standardize(frame: pd.DataFrame, features: list[str]) -> tuple[np.ndarray, pd.Series, pd.Series]:
    """Standardize the specified features in the DataFrame and return the standardized values along with the means and standard deviations used for standardization."""
    means = frame[features].mean()
    stds = frame[features].std(ddof=0).replace(0, 1)
    standardized = (frame[features] - means) / stds
    return standardized.to_numpy(dtype=float), means, stds


def fit_ols(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Fit an ordinary least squares regression model to the data and return the coefficients and predictions."""
    design = np.column_stack([np.ones(len(x)), x])
    coefficients, *_ = np.linalg.lstsq(design, y, rcond=None)
    predictions = design @ coefficients
    return coefficients, predictions


def regression_metrics(y: np.ndarray, predictions: np.ndarray) -> dict[str, float]:
    """Calculate regression metrics (R-squared, RMSE, MAE) based on the true values and predictions."""
    residuals = y - predictions
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((y - y.mean()) ** 2))
    return {
        "r_squared": 1 - sse / sst,
        "rmse": float(np.sqrt(np.mean(residuals**2))),
        "mae": float(np.mean(np.abs(residuals))),
    }

## Load Data and Fit Model
Read the joined CSV, filter to the analysis sample, standardize features, and fit the OLS regression model.

In [16]:
ensure_figures_dir()
joined = pd.read_csv(JOINED_FILE)
analysis = joined[joined["analysis_sample"].astype(bool)].copy()
features = [feature for feature in MODEL_FEATURES if feature in analysis.columns]
model_data = analysis.dropna(subset=features + [TARGET]).copy()

x, feature_means, feature_stds = standardize(model_data, features)
y = model_data[TARGET].to_numpy(dtype=float)
coefficients, predictions = fit_ols(x, y)
metrics = regression_metrics(y, predictions)

model_data["pred_salary_cap_pct"] = predictions
model_data["residual_actual_minus_pred"] = model_data[TARGET] - model_data["pred_salary_cap_pct"]
model_data["abs_residual"] = model_data["residual_actual_minus_pred"].abs()

print(f"Rows used: {len(model_data):,}")
print(f"R-squared: {metrics['r_squared']:.3f}")
print(f"RMSE: {metrics['rmse']:.3f}")
print(f"MAE: {metrics['mae']:.3f}")

Rows used: 2,973
R-squared: 0.601
RMSE: 5.577
MAE: 4.286


## Build Coefficient Table and Summary
Store standardized coefficients and model metrics in DataFrames for export and review.

In [17]:
# Create a table of standardized coefficients along with feature means and standard deviations
coefficient_table = pd.DataFrame(
    {
        "feature": features,
        "standardized_coefficient": coefficients[1:],
        "feature_mean": feature_means.values,
        "feature_std": feature_stds.values,
    }
).sort_values("standardized_coefficient", key=lambda col: col.abs(), ascending=False)

# Create a summary table of regression metrics and model information
summary = pd.DataFrame(
    [
        ("rows", len(model_data)),
        ("target", TARGET),
        ("features", ", ".join(features)),
        ("intercept", coefficients[0]),
        ("r_squared", metrics["r_squared"]),
        ("rmse", metrics["rmse"]),
        ("mae", metrics["mae"]),
    ],
    columns=["metric", "value"],
)

# Prepare a table of results with relevant columns for analysis and sorting by residuals
result_columns = [
    'player',
    'season',
    'team',
    'salary',
    'salary_cap_pct',
    'salary_tier',
    'is_contract_year',
    'age',
    'gp',
    'min',
    'pie',
    'ts_pct',
    'usg_pct',
    'fgm_pg',
    'fg_pct',
    'pred_salary_cap_pct',
    'residual_actual_minus_pred',
    'abs_residual',
]
result_columns = [column for column in result_columns if column in model_data.columns]
results = model_data[result_columns].sort_values("abs_residual", ascending=False)

over_expected = results.sort_values("residual_actual_minus_pred", ascending=False).head(20)
under_expected = results.sort_values("residual_actual_minus_pred", ascending=True).head(20)

coefficient_table.head()

,feature,standardized_coefficient,feature_mean,feature_std
6,net_rating,10.300807,0.119643,5.999444
4,off_rating,-8.707200,110.222133,5.057892
5,def_rating,7.456589,110.103263,4.335327
12,fgm_pg,5.881454,4.325563,2.239304
0,age,3.056624,26.629667,4.295167


## Save Results to CSV
Export model results, top over/under-paid players, regression summary, and coefficients to CSV files.

In [22]:
results.to_csv(MODEL_RESULTS_FILE, index=False)
over_expected.to_csv(TOP_OVER_EXPECTED_FILE, index=False)
under_expected.to_csv(TOP_UNDER_EXPECTED_FILE, index=False)
summary.to_csv(REGRESSION_SUMMARY_FILE, index=False)
coefficient_table.to_csv(REGRESSION_COEFFICIENTS_FILE, index=False)

print(f"Saved model results: {MODEL_RESULTS_FILE.name} ({len(results):,} rows)")
print(f"Saved top paid-over-model rows: {TOP_OVER_EXPECTED_FILE.name}")
print(f"Saved top paid-under-model rows: {TOP_UNDER_EXPECTED_FILE.name}")
print(f"Saved regression summary: {REGRESSION_SUMMARY_FILE.name}")

Saved model results: nba_model_results.csv (2,973 rows)
Saved top paid-over-model rows: top_over_expected.csv
Saved top paid-under-model rows: top_under_expected.csv
Saved regression summary: regression_summary.csv


## Figure 7: Predicted vs Actual Salary Cap Percentage
Plot predicted salary cap percentage against actual values. Points close to the diagonal line indicate accurate predictions.

In [19]:
def save_predicted_vs_actual(results: pd.DataFrame) -> None:
    """Create and save a scatter plot comparing predicted vs actual salary cap percentages, with a reference line for perfect predictions."""
    plt.figure(figsize=(11, 6))
    plt.scatter(
        results['pred_salary_cap_pct'],
        results[TARGET],
        s=26,
        alpha=0.55,
        color='#2563EB',
        edgecolors='none',
    )
    low = min(results['pred_salary_cap_pct'].min(), results[TARGET].min())
    high = max(results['pred_salary_cap_pct'].max(), results[TARGET].max())
    plt.plot([low, high], [low, high], color='#111827', linewidth=2, label='Actual = predicted')
    r_squared = np.corrcoef(results['pred_salary_cap_pct'], results[TARGET])[0, 1] ** 2
    plt.annotate(
        f'$R^2 = {r_squared:.3f}$',
        xy=(0.04, 0.90),
        xycoords='axes fraction',
        fontsize=9,
    )
    plt.title('Predicted vs Actual Salary Cap Percentage', fontsize=13)
    plt.xlabel('Predicted salary cap percentage', fontsize=10)
    plt.ylabel('Actual salary cap percentage', fontsize=10)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '07_predicted_vs_actual_salary.png', dpi=180, bbox_inches='tight')
    plt.close()

save_predicted_vs_actual(model_data)

## Figure 8: Standardized Regression Coefficients (EDA only, not used in report)
Show which features have the strongest positive or negative relationship with salary cap percentage after standardization. This was exploratory and not included in the final report.

In [20]:
def save_coefficient_plot(coefficients: pd.DataFrame) -> None:
    """Create and save a horizontal bar plot of standardized regression coefficients, colored by sign and sorted by magnitude."""
    coefficients = coefficients.sort_values("standardized_coefficient")
    colors = np.where(coefficients["standardized_coefficient"].ge(0), "#2563EB", "#DC2626")

    plt.figure(figsize=(8, 6))
    plt.barh(coefficients["feature"], coefficients["standardized_coefficient"], color=colors)
    plt.axvline(0, color="#111827", linewidth=1)
    plt.title("Standardized Linear Regression Coefficients")
    plt.xlabel("Coefficient")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "08_standardized_coefficients.png", dpi=180, bbox_inches="tight")
    plt.close()

save_coefficient_plot(coefficient_table)

## Figure 9: Largest Salary Residuals
Show the players whose actual salary was furthest from the model's prediction. Red bars are overpaid relative to stats, blue bars are underpaid.

In [21]:
def save_residual_outlier_plot(results: pd.DataFrame) -> None:
    """Create and save a horizontal bar plot of the largest positive and negative residuals (actual minus predicted salary cap percentage), colored by sign."""
    over = results.sort_values('residual_actual_minus_pred', ascending=False).head(5).copy()
    under = results.sort_values('residual_actual_minus_pred', ascending=True).head(5).copy()
    plot_data = pd.concat([under, over], ignore_index=True)
    plot_data['label'] = plot_data['player'] + ' (' + plot_data['season'] + ')'
    plot_data = plot_data.sort_values('residual_actual_minus_pred')
    colors = np.where(plot_data['residual_actual_minus_pred'].ge(0), '#DC2626', '#2563EB')

    plt.figure(figsize=(12, 5))
    plt.barh(plot_data['label'], plot_data['residual_actual_minus_pred'], color=colors)
    plt.axvline(0, color='#111827', linewidth=1.2)
    plt.title('Largest Salary Residuals from Simple Linear Model', fontsize=13)
    plt.xlabel('Actual minus predicted salary cap percentage', fontsize=10)
    plt.ylabel('', fontsize=10)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '09_top_salary_residuals.png', dpi=180, bbox_inches='tight')
    plt.close()

save_residual_outlier_plot(results)